In [ ]:
import pandas as pd
import numpy as np


n_registros = 50

dados = {

"temperatura_interna": np.random.uniform(20.0,45.0, n_registros),
"temperatura_externa": np.random.uniform(-10.0, 30.0, n_registros),
"integridade_estrutural": np.random.choice([0,1], size=n_registros, p=[0.05,0.95]),
"nivel_energia": np.random.uniform(10.0, 100.0, n_registros),
"pressão_tanques": np.random.uniform(150.0,250.0,n_registros),
"modulo_critico": np.random.choice([0,1], size=n_registros, p=[0.02,0.98])
}

df_telemetria = pd.DataFrame(dados)

display(df_telemetria.head())




In [ ]:
capacidade_total_kwh = 80.0
consumo_lancamento_kwh = 12.0
taxa_perdas = 0.08

df_telemetria["energia_atual_kwh"] = capacidade_total_kwh * (df_telemetria["nivel_energia"] / 100.0)
df_telemetria["consumo_total_kwh"] = consumo_lancamento_kwh * (1.0 + taxa_perdas)
df_telemetria["autonomia_residual_kwh"] = df_telemetria["energia_atual_kwh"] - df_telemetria["consumo_total_kwh"]




resultados = []

for indice, linha in df_telemetria.iterrows():
  temp_interna = linha["temperatura_interna"]
  temp_externa = linha["temperatura_externa"]
  integ_est = linha["integridade_estrutural"]
  nvl_energia = linha["nivel_energia"]
  press_tanque = linha["pressão_tanques"]
  modulo = linha["modulo_critico"]
  autonomia_residual = linha["autonomia_residual_kwh"]

  if (temp_interna < 40) and (temp_externa) <= 30 and (integ_est == 1) and (nvl_energia >= 20) and (press_tanque >= 150) and (modulo == 1) and (autonomia_residual > 5.0):
    decisao = "Pronto para decolagem"
  else:
    decisao = "Abortar decolagem"

  resultados.append(decisao)


df_telemetria["decisao_lancamento"] = resultados 


condicoes_ia = [
    (df_telemetria["decisao_lancamento"] == "Pronto para decolagem"),
    (df_telemetria["autonomia_residual_kwh"] <= 5.0) | (df_telemetria["integridade_estrutural"] == 0),
    (df_telemetria["temperatura_interna"] >= 40.0) | (df_telemetria["pressão_tanques"] < 150.0)
]
categorias_ia = ["Nominal", "Crítico (Segurança/Estrutura)", "Alerta (Térmico/Pressão)"]
df_telemetria["status_operacional"] = np.select(condicoes_ia, categorias_ia, default="Em Observação")

falhas_temp = (df_telemetria["temperatura_interna"] >= 40.0).sum()
falhas_pressao = (df_telemetria["pressão_tanques"] < 150.0).sum()
falhas_energia = (df_telemetria["autonomia_residual_kwh"] <= 5.0).sum()
total_abortadas = (df_telemetria["decisao_lancamento"] == "Abortar decolagem").sum()
taxa_risco = (total_abortadas / len(df_telemetria)) * 100


print("\n" + "="*60)
print("  RELATÓRIO DE TELEMETRIA  ")
print("\n" + "="*60)

print("\nResumo das avaliações")
print(df_telemetria["decisao_lancamento"].value_counts())

print("\n" + "-"*60)
print("ANÁLISE ENERGÉTICA (MÉDIAS DA SIMULAÇÃO):")
print(f" • Capacidade Total do Sistema : {capacidade_total_kwh:.2f} kWh")
print(f" • Carga Atual Média na Bateria: {df_telemetria['nivel_energia'].mean():.2f}% ({df_telemetria['energia_atual_kwh'].mean():.2f} kWh)")
print(f" • Consumo Estimado de Decolagem : {df_telemetria['consumo_total_kwh'].iloc[0]:.2f} kWh (com perdas)")
print(f" • Autonomia Residual Média    : {df_telemetria['autonomia_residual_kwh'].mean():.2f} kWh")


print("\n" + "-"*60)
print("DIAGNÓSTICO E SUGESTÕES DE RISCO (IA):")
print(f" • Índice de Risco da Missão: {taxa_risco:.1f}% de rejeição nos testes.")

if falhas_temp > 5:
    print(" ⚠️ Sugestão: Alta incidência térmica. Revisar refrigeração interna.")
if falhas_energia > 5:
    print(" ⚠️ Sugestão: Margem energética restrita. Avaliar expandir baterias.")
if taxa_risco < 20:
    print(" ✅ Sugestão: O sistema demonstra alta estabilidade operacional.")



print("\n" + "="*60)
print("VEREDITO FINAL DO LANÇAMENTO:")

if (df_telemetria["decisao_lancamento"] == "Abortar decolagem").any():
  print("❌ LANÇAMENTO ABORTADO")
  print("   (Foram detectadas instabilidades nos sensores ou margem energética insuficiente)")
else:
  print("🚀 LANÇAMENTO APROVADO")
  print("   (Todos os parâmetros  e reservas energéticas permaneceram seguros)")

print("=" * 60)
